## The Clean Way: Python Classes

In [44]:
import time


class OrderProcessor:
    """Manages the lifecycle of an e-commerce order."""

    def __init__(self, order_id: int, items: list):
        # Shared state lives here inside 'self'
        self.order_id = order_id
        self.items = items
        self.status = "Created"
        self.total_price = 0.0

    def calculate_total(self, tax_rate: float = 0.08) -> float:
        """Calculates the total cost of items including tax."""
        subtotal = sum(item["price"] for item in self.items)
        self.total_price = round(subtotal * (1 + tax_rate), 2)
        return self.total_price

    def process_payment(self) -> bool:
        """Simulates a payment gateway charge."""
        if self.total_price == 0:
            # Prevent processing before calculating the total
            self.calculate_total()

        print(f"Charging card for Order #{self.order_id}: ${self.total_price}")
        # Simulating a successful network call
        time.sleep(0.5)
        self.status = "Paid"
        return True

    def ship_order(self) -> str:
        """Updates shipping status if the order is paid."""
        if self.status != "Paid":
            return f"Order {self.order_id} cannot ship. Status: {self.status}"

        self.status = "Shipped"
        return f"Order {self.order_id} is on its way to the customer!"

In [45]:
# --- Example Usage ---

# Sample data
cart = [{"name": "Laptop", "price": 1000}, {"name": "Mouse", "price": 50}]

# Create an instance of the class
my_order = OrderProcessor(order_id=404, items=cart)

# Execute the workflow seamlessly without passing variables back and forth
my_order.calculate_total(tax_rate=0.10)
my_order.process_payment()
shipping_alert = my_order.ship_order()

print(shipping_alert)

Charging card for Order #404: $1155.0
Order 404 is on its way to the customer!


## The messy way


In [3]:
import time

# --- Global State Problem ---
# In a real app, global variables get tangled fast.
# If two orders run at once, they overwrite each other!
current_order_id = None
current_items = []
current_status = "Unknown"
current_total = 0.0

In [25]:
def init_order(order_id: int, items: list):
    """Initializes the global order state."""
    global current_order_id, current_items, current_status, current_total
    current_order_id = order_id
    current_items = items
    current_status = "Created"
    current_total = 0.0

In [24]:
def calculate_total(tax_rate: float = 0.08):
    """Calculates total and updates a global variable."""
    if not current_items:
        print("Error: No items found!")
        return None

    subtotal = sum(item["price"] for item in current_items)
    current_total = round(subtotal * (1 + tax_rate), 2)
    return current_total

In [27]:
current_total = calculate_total(tax_rate=0.10)

In [36]:
def process_payment():
    """Processes payment but relies heavily on the correct sequence of globals."""
    # What if calculate_total wasn't called first? The total is $0.0!
    print(f"Charging card for Order #{current_order_id}: ${current_total}")

    time.sleep(0.5)
    current_status = "Paid"
    return current_status

In [37]:
process_payment()

Charging card for Order #505: $0.0


'Paid'

In [40]:
def ship_order(current_status: str):
    """Ships the order, checking the global status flag."""
    if current_status != "Paid":
        return (
            f"Error: Order {current_order_id} cannot ship. Status: {current_status}"
        )

    current_status = "Shipped"
    return f"Order {current_order_id} shipped successfully!"

In [42]:
# --- Running the Messy Code ---

cart = [{"name": "Laptop", "price": 1000}, {"name": "Mouse", "price": 50}]

# We have to initialize the state manually
init_order(404, cart)

# Everything looks okay if we call them in perfect order...
current_total = calculate_total(tax_rate=0.10)
current_status = process_payment()
print(ship_order(current_status))

# --- THE BREAKING POINT ---
# What happens if we start a new order workflow halfway through?
# Order 505 completely overwrites the global state of Order 404!
init_order(505, [{"name": "Headphones", "price": 100}])
print(
    f"\n[CRASH RISK] Current global order ID is now: {current_order_id}"
)

Charging card for Order #404: $1155.0
Order 404 shipped successfully!

[CRASH RISK] Current global order ID is now: 505
